In [1]:
from __future__ import annotations

import os
import sys

import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

sys.path.append(r'C:\Users\Stefa\Documents\studio-paper')


from parfitt_trb import TRBConfig, penetration_vs_actual, pwsd, run_trb
from parfitt_trb import plots
from parfitt_trb.display import rollup_ratio
from parfitt_trb.local_spark import build_local_spark
from examples.synth import simulate_panel

OUT = os.path.join(r'C:\Users\Stefa\Documents\studio-paper\examples', "figuresSte")
os.makedirs(OUT, exist_ok=True)

In [2]:
print(sys.executable)

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

c:\Users\Stefa\Documents\studio-paper\.venv\Scripts\python.exe


In [ ]:
PROMO_WEEK = 18
pdf = simulate_panel(
    n_households=6000, weeks=40, K=0.24, a=0.22,
    rbr_start=0.42, rbr_stable=0.24, cat_interval=2, seed=11,
    promo_week=PROMO_WEEK, promo_K=0.12, promo_rbr=0.07
)
pdf["ISOWEEKYEAR"] = pd.to_datetime(pdf["txn_date"]).dt.strftime("%G-%V")

spark = build_local_spark("trb-usage-template")
df = spark.createDataFrame(pdf)            # in production: your cluster DataFrame
cfg = TRBConfig(launch_date="2024-01-01", period_length_days=14,
                buying_index_base="triers")

In [ ]:
df.limit(10).show()

In [ ]:
cfg = TRBConfig(
    launch_date="2024-01-01"
    # , period_length_days=14
    , bucket_column='ISOWEEKYEAR'
    , buying_index_base="triers"
    , buying_index_window_days = 30
    , rbr_interval_mode="bucket"
    , rbr_bucket_unit="month"         # RBR su intervalli mensili
)

In [ ]:
res = run_trb(df, cfg)

In [ ]:
pen = res.penetration

In [ ]:
print(pen)

print([(res.label(t), p) for t, p in res.penetration.series] )
print(pen.ultimate_penetration)
print(pen.snapshot)


In [ ]:
from parfitt_trb.aggregation import SparkAggregator, _build_bucket_map, _period_col, _cohort_col, _max_interval_col
from pyspark.sql import functions as F

agg = SparkAggregator(df, cfg)

In [ ]:
p = agg._prepare(df)
p.limit(10).show()
p = p.filter(F.col("ts") <= F.lit(agg._adate).cast("date"))

In [ ]:
agg._bucket_to_period
agg._origin

obs_filter = F.col("ts") >= F.lit(agg._origin).cast("date")

first = (
    agg._p.filter(obs_filter)
    .groupBy("bucket").agg(F.min("ts").alias("first")).toPandas()
    .set_index("bucket")["first"]
)

_build_bucket_map(first)

agg._build        

In [ ]:
cfg.rbr_interval_mode

In [ ]:
tr = (
    agg._p.filter(F.col("is_brand"))
    .filter(F.col("ts") >= F.lit(agg._origin).cast("date"))
    .groupBy("card").agg(F.min(F.struct("ts", "bucket")).alias("m"))
    .select("card", F.col("m.ts").alias("trial_ts"), F.col("m.bucket").alias("bucket"))
)
origin = F.lit(agg._origin).cast("date")

tr = tr.withColumn("entry_week", F.floor(F.datediff(F.col("trial_ts"), origin) / 7) + 1)
tr = tr.withColumn("entry_period", _period_col(
    F, F.col("trial_ts"), cfg, agg._origin, agg._origin_month,
    agg._bucket_to_period, "bucket"))
tr = tr.withColumn("cohort", _cohort_col(
    F, F.col("entry_week"), cfg.cohort_boundaries_weeks, cfg.include_prelaunch_cohort))
tr = tr.withColumn("max_interval", _max_interval_col(
    F, F.col("trial_ts"), agg._adate, agg._adate_month, cfg))
# keep = ["card", "trial_ts", "entry_week", "entry_period", "cohort", "max_interval"]

tr.show()